Benchmark for the kidney dataset

In [1]:
import sys
from pathlib import Path
import os
import scanpy as sc
import numpy as np

REPO_ROOT = Path("~/scRNA-cross-donor-generalization").expanduser()
os.chdir(REPO_ROOT)
sys.path.append(str(REPO_ROOT / "src"))

In [2]:
from pathlib import Path
import scanpy as sc

adata_path = Path("data/kidney/kidney.h5ad")
adata = sc.read_h5ad(adata_path)

print(adata)

print("\nobs columns:")
print(adata.obs.columns.tolist())

print("\nvar columns:")
print(adata.var.columns.tolist())

print("\nobsm keys:")
print(list(adata.obsm.keys()))

for col in [
    "donor_id",
    "cell_type",
    "cell_type_original",
    "author_cell_type",
    "assay",
    "suspension_type",
    "tissue",
    "tissue_general",
    "disease",
    "sample_id",
    "batch",
]:
    if col in adata.obs.columns:
        print(f"\n{col}:")
        print(adata.obs[col].value_counts(dropna=False).head(20))

AnnData object with n_obs × n_vars = 27675 × 26791
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sampletype', 'CMVstatus', 'percent.mt', 'Group', 'Female', 'Male', 'sampleType', 'log10GenesPerUMI', 'nCount_SCT', 'nFeature_SCT', 'SCT_snn_res.0.8', 'seurat_clusters', 'S.Score', 'G2M.Score', 'Phase', 'percent.ribo', 'study_pi', 'sample_id', 'donor_id', 'protocol_url', 'institute', 'library_id', 'library_id_repository', 'manner_of_death', 'sample_source', 'sex_ontology_term_id', 'sample_collection_method', 'tissue_type', 'sampled_site_condition', 'tissue_ontology_term_id', 'tissue_free_text', 'sample_preservation_method', 'suspension_type', 'cell_enrichment', 'assay_ontology_term_id', 'sequenced_fragment', 'sequencing_platform', 'is_primary_data', 'reference_genome', 'alignment_software', 'intron_inclusion', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'library_preparation_batch', 'library_sequencing_run', 'gene_annot

In [3]:
for col in [
    "donor_id",
    "sample_id",
    "library_id",
    "library_preparation_batch",
    "library_sequencing_run",
    "sample_source",
    "cell_enrichment",
    "sampled_site_condition",
    "sex",
    "disease",
    "tissue",
]:
    if col in adata.obs:
        print(f"\n{col}")
        print(adata.obs[col].value_counts(dropna=False))


donor_id
donor_id
HKB15    4738
HKB17    3739
HKB18    3280
HKB16    2366
HKB33    1894
HKB32    1811
HKB34    1264
HKB24    1182
HKB23    1102
HKB14    1080
HKB27    1041
HKB13     938
HKB28     712
HKB31     606
HKB10     584
HKB26     553
HKB21     344
HKB11     315
HKB19     126
Name: count, dtype: int64

sample_id
sample_id
Total2     4738
Total4     3739
Total5     3280
Total3     2366
Total8     1894
Total7     1811
Total9     1264
CD45_7     1182
CD45_6     1102
Total1     1080
CD45_9     1041
CD45_3      938
CD45_10     712
Total6      606
CD45_1      584
CD45_8      553
CD45_5      344
CD45_2      315
CD45_4      126
Name: count, dtype: int64

library_id
library_id
Total2     4738
Total4     3739
Total5     3280
Total3     2366
Total8     1894
Total7     1811
Total9     1264
CD45_7     1182
CD45_6     1102
Total1     1080
CD45_9     1041
CD45_3      938
CD45_10     712
Total6      606
CD45_1      584
CD45_8      553
CD45_5      344
CD45_2      315
CD45_4      126
Name: count

In [4]:
import pandas as pd

cols = ["donor_id", "sample_id", "library_id", "library_preparation_batch", "cell_enrichment"]
cols = [c for c in cols if c in adata.obs.columns]

donor_meta = (
    adata.obs[cols]
    .drop_duplicates()
    .sort_values("donor_id")
)

print(donor_meta.to_string(index=False))

donor_id sample_id library_id library_preparation_batch cell_enrichment
   HKB10    CD45_1     CD45_1                   batch01     CL:0000738+
   HKB11    CD45_2     CD45_2                   batch02     CL:0000738+
   HKB13    CD45_3     CD45_3                   batch03     CL:0000738+
   HKB14    Total1     Total1                   batch11              na
   HKB15    Total2     Total2                   batch12              na
   HKB16    Total3     Total3                   batch13              na
   HKB17    Total4     Total4                   batch14              na
   HKB18    Total5     Total5                   batch15              na
   HKB19    CD45_4     CD45_4                   batch04     CL:0000738+
   HKB21    CD45_5     CD45_5                   batch05     CL:0000738+
   HKB23    CD45_6     CD45_6                   batch06     CL:0000738+
   HKB24    CD45_7     CD45_7                   batch07     CL:0000738+
   HKB26    CD45_8     CD45_8                   batch08     CL:0

In [5]:
print("CELLxGENE cell_type:")
print(adata.obs["cell_type"].value_counts())

print("\nauthor_cell_type:")
print(adata.obs["author_cell_type"].value_counts())

CELLxGENE cell_type:
cell_type
epithelial cell of proximal tubule                                    20983
kidney loop of Henle cortical thick ascending limb epithelial cell     1156
kidney loop of Henle descending limb epithelial cell                   1011
mononuclear phagocyte                                                   999
T cell                                                                  938
kidney capillary endothelial cell                                       572
natural killer cell                                                     466
renal principal cell                                                    455
renal alpha-intercalated cell                                           304
mesangial cell                                                          130
renal beta-intercalated cell                                            130
kidney connecting tubule epithelial cell                                122
kidney distal convoluted tubule epithelial cell          

In [6]:
mapping = (
    adata.obs[["author_cell_type", "cell_type"]]
    .drop_duplicates()
    .sort_values(["cell_type", "author_cell_type"])
)

print(mapping.to_string(index=False))

author_cell_type                                                          cell_type
               B                                                             B cell
         CCDlike                      kidney cortex collecting duct epithelial cell
             CNT                           kidney connecting tubule epithelial cell
            CTAL kidney loop of Henle cortical thick ascending limb epithelial cell
             DCT                    kidney distal convoluted tubule epithelial cell
     Endothelial                                  kidney capillary endothelial cell
             ICA                                      renal alpha-intercalated cell
             ICB                                       renal beta-intercalated cell
         LOHlike               kidney loop of Henle descending limb epithelial cell
       Mesangial                                                     mesangial cell
             MNP                                              mononuclear ph

In [7]:
ct = pd.crosstab(adata.obs["cell_type"], adata.obs["donor_id"])

summary = pd.DataFrame({
    "n_cells": ct.sum(axis=1),
    "n_donors": (ct > 0).sum(axis=1),
    "median_cells_per_present_donor": ct.replace(0, pd.NA).median(axis=1),
    "min_cells_per_present_donor": ct.replace(0, pd.NA).min(axis=1),
}).sort_values(["n_donors", "n_cells"], ascending=False)

print(summary)

                                                    n_cells  n_donors  \
cell_type                                                               
epithelial cell of proximal tubule                    20983        19   
kidney loop of Henle descending limb epithelial...     1011        19   
kidney capillary endothelial cell                       572        19   
renal alpha-intercalated cell                           304        19   
kidney loop of Henle cortical thick ascending l...     1156        18   
renal principal cell                                    455        18   
renal beta-intercalated cell                            130        18   
kidney distal convoluted tubule epithelial cell         109        18   
mononuclear phagocyte                                   999        16   
mesangial cell                                          130        16   
parietal epithelial cell                                 64        15   
kidney connecting tubule epithelial cell           

In [8]:
min_cells = 200
min_donors = 5

keep_celltypes = summary.query(
    "n_cells >= @min_cells and n_donors >= @min_donors"
).index.tolist()

print("Kept cell types:", len(keep_celltypes))
print(keep_celltypes)

adata_filt = adata[adata.obs["cell_type"].isin(keep_celltypes)].copy()
print(adata_filt)
print(adata_filt.obs["cell_type"].value_counts())
print(adata_filt.obs["donor_id"].value_counts())

Kept cell types: 9
['epithelial cell of proximal tubule', 'kidney loop of Henle descending limb epithelial cell', 'kidney capillary endothelial cell', 'renal alpha-intercalated cell', 'kidney loop of Henle cortical thick ascending limb epithelial cell', 'renal principal cell', 'mononuclear phagocyte', 'T cell', 'natural killer cell']
AnnData object with n_obs × n_vars = 26884 × 26791
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sampletype', 'CMVstatus', 'percent.mt', 'Group', 'Female', 'Male', 'sampleType', 'log10GenesPerUMI', 'nCount_SCT', 'nFeature_SCT', 'SCT_snn_res.0.8', 'seurat_clusters', 'S.Score', 'G2M.Score', 'Phase', 'percent.ribo', 'study_pi', 'sample_id', 'donor_id', 'protocol_url', 'institute', 'library_id', 'library_id_repository', 'manner_of_death', 'sample_source', 'sex_ontology_term_id', 'sample_collection_method', 'tissue_type', 'sampled_site_condition', 'tissue_ontology_term_id', 'tissue_free_text', 'sample_preservation_method', 'suspension_type', 'cell_enric

In [9]:
for col in ["cell_enrichment", "sample_source", "sampletype", "sampleType", "sampled_site_condition"]:
    if col in adata.obs:
        print(f"\n{col}")
        print(adata.obs[col].value_counts(dropna=False))


cell_enrichment
cell_enrichment
na             20778
CL:0000738+     6897
Name: count, dtype: int64

sample_source
sample_source
living organ donor    27675
Name: count, dtype: int64

sampletype
sampletype
NaN    20778
LD      6897
Name: count, dtype: int64

sampleType
sampleType
LD    27675
Name: count, dtype: int64

sampled_site_condition
sampled_site_condition
healthy    27675
Name: count, dtype: int64


In [10]:
pd.crosstab(adata.obs["donor_id"], adata.obs["cell_enrichment"])

cell_enrichment,CL:0000738+,na
donor_id,,
HKB10,584,0
HKB11,315,0
HKB13,938,0
HKB14,0,1080
HKB15,0,4738
HKB16,0,2366
HKB17,0,3739
HKB18,0,3280
HKB19,126,0


In [11]:
pd.crosstab(adata.obs["cell_type"], adata.obs["cell_enrichment"])

cell_enrichment,CL:0000738+,na
cell_type,,
B cell,58,10
kidney cortex collecting duct epithelial cell,4,41
kidney connecting tubule epithelial cell,76,46
kidney loop of Henle cortical thick ascending limb epithelial cell,642,514
kidney distal convoluted tubule epithelial cell,51,58
kidney capillary endothelial cell,252,320
renal alpha-intercalated cell,106,198
renal beta-intercalated cell,49,81
kidney loop of Henle descending limb epithelial cell,82,929


In [12]:
import numpy as np
from scipy import sparse

X = adata.X
vals = X.data if sparse.issparse(X) else X.ravel()
vals_sample = vals[:100000]

print("min:", vals_sample.min())
print("max:", vals_sample.max())
print("mean:", vals_sample.mean())
print("integer-like:", np.allclose(vals_sample, np.round(vals_sample)))

min: 0.0
max: 4553.716
mean: 4.6264234
integer-like: False
